<a href="https://colab.research.google.com/github/witchdrmd/CS598DLH/blob/main/SDOH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📦 STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 📁 STEP 2: Define Paths (edit if needed)
mimic_dir = '/content/drive/MyDrive/mimic'
noteevents_path = f'{mimic_dir}/NOTEEVENTS.csv.gz'
sbdh_path = f'{mimic_dir}/MIMIC-SBDH.csv'
output_path = f'{mimic_dir}/alcohol_use_notes.csv'

In [ ]:
!pip install --upgrade --force-reinstall numpy pandas

  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalli

In [ ]:
!pip install transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.


In [ ]:
import pandas as pd

In [ ]:
# Load NOTEEVENTS (full file, compressed)
notes_df = pd.read_csv(noteevents_path, compression='gzip', low_memory=False)

# Load SBDH annotations
sbdh_df = pd.read_csv(sbdh_path)

print("NOTEEVENTS columns:", notes_df.columns.tolist())
print("MIMIC-SBDH columns:", sbdh_df.columns.tolist())

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
# 🔍 STEP 4: Merge on row_id → ROW_ID
merged_df = pd.merge(sbdh_df, notes_df, left_on="row_id", right_on="ROW_ID", how="inner")

# Filter to Discharge Summaries only
merged_df = merged_df[merged_df["CATEGORY"] == "Discharge summary"]

# Keep only alcohol-relevant fields
merged_df = merged_df[["SUBJECT_ID", "HADM_ID", "TEXT", "behavior_alcohol"]]
merged_df = merged_df.rename(columns={"TEXT": "note", "behavior_alcohol": "label"})

print("Sample merged data:\n", merged_df.sample(3))

NameError: name 'sbdh_df' is not defined

In [ ]:
merged_df.to_csv(output_path, index=False)
print(f"Saved cleaned Alcohol Use notes to: {output_path}")

Saved cleaned Alcohol Use notes to: /content/drive/MyDrive/mimic/alcohol_use_notes.csv


In [ ]:
alcohol_df = pd.read_csv('/content/drive/MyDrive/mimic/alcohol_use_notes.csv')
print(alcohol_df.sample(3))

      SUBJECT_ID   HADM_ID                                               note  \
5081       87989  118854.0  Admission Date:  [**2117-8-17**]              ...   
3524       71792  190178.0  Admission Date:  [**2201-2-3**]              D...   
6712       70795  133651.0  Admission Date:  [**2113-10-9**]              ...   

      label  
5081      3  
3524      0  
6712      0  


In [ ]:
import medspacy
from medspacy.section_detection import Sectionizer

# Load medspacy pipeline
nlp = medspacy.load()

# Make sure to remove if sectionizer already exists
if "sectionizer" in nlp.pipe_names:
    nlp.remove_pipe("sectionizer")

# Add the sectionizer using 'medspacy_sectionizer'
nlp.add_pipe("medspacy_sectionizer", config={"rules": "default"}, last=True) # changed from sectionizer object to 'medspacy_sectionizer' string

# Confirm it worked
print("Pipeline components:", nlp.pipe_names)

Pipeline components: ['medspacy_pyrush', 'medspacy_target_matcher', 'medspacy_context', 'medspacy_sectionizer']


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/mimic/alcohol_use_notes.csv')
df = df.dropna(subset=["note", "label"])

In [ ]:
def extract_social_history(text):
    doc = nlp(text)
    for section in doc._.sections:
        if section.category and "social_history" in section.category.lower():
            start, end = section.body_span
            return doc.text[start:end]
    return None

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/mimic/alcohol_use_notes.csv')
batch_size = 500
num_batches = len(df) // batch_size + 1

results = []

for i in tqdm(range(num_batches)):
    start = i * batch_size
    end = min((i + 1) * batch_size, len(df))
    batch = df.iloc[start:end].copy()
    batch["social_note"] = batch["note"].apply(extract_social_history)
    batch = batch.dropna(subset=["social_note"])
    results.append(batch)
    print(f"✅ Batch {i+1}/{num_batches} complete")

  7%|▋         | 1/15 [06:20<1:28:53, 380.94s/it]

✅ Batch 1/15 complete


 13%|█▎        | 2/15 [12:02<1:17:34, 358.06s/it]

✅ Batch 2/15 complete


 20%|██        | 3/15 [17:43<1:09:58, 349.87s/it]

...

✅ Batch 13/15 complete


 93%|█████████▎| 14/15 [1:23:20<06:00, 360.63s/it]

✅ Batch 14/15 complete


100%|██████████| 15/15 [1:23:39<00:00, 334.65s/it]

✅ Batch 15/15 complete


In [ ]:
df_final = pd.concat(results).reset_index(drop=True)
df_final["note"] = df_final["social_note"]
df_final = df_final.drop(columns=["social_note"])

df_final.to_csv('/content/drive/MyDrive/mimic/alcohol_use_social_only.csv', index=False)
print("✅ Final cleaned dataset saved.")

✅ Final cleaned dataset saved.


In [ ]:
print(f"✅ df_final has {len(df_final)} rows")
print(df_final.columns.tolist())
df_final.sample(3)
print(df_final.isnull().sum())
print(df_final["label"].value_counts())


NameError: name 'df_final' is not defined

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/mimic/alcohol_use_social_only.csv')
df = df.dropna(subset=["note", "label"])
df["label"] = df["label"].astype(str).astype(int)  # Forces full conversion even if strings sneak in
print(df["label"].value_counts())

label
3    2443
1    2077
0    1657
2     515
4     332
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X_train, X_test, y_train, y_test = train_test_split(
    df["note"], df["label"], test_size=0.2, stratify=df["label"], random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, class_weight="balanced", random_state=42
)
rf.fit(X_train_vec, y_train)
rf_preds = rf.predict(X_test_vec)
print("🎯 Random Forest Results:\n", classification_report(y_test, rf_preds))

🎯 Random Forest Results:
               precision    recall  f1-score   support

           0       0.29      0.68      0.41       331
           1       0.39      0.24      0.30       416
           2       0.12      0.11      0.11       103
           3       0.41      0.20      0.27       489
           4       0.11      0.09      0.10        66

    accuracy                           0.31      1405
   macro avg       0.27      0.26      0.24      1405
weighted avg       0.34      0.31      0.29      1405



In [ ]:
xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)
xgb.fit(X_train_vec, y_train)
xgb_preds = xgb.predict(X_test_vec)
print("🎯 XGBoost Results:\n", classification_report(y_test, xgb_preds))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:16:17] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


🎯 XGBoost Results:
               precision    recall  f1-score   support

           0       0.28      0.09      0.14       331
           1       0.37      0.29      0.32       416
           2       0.21      0.03      0.05       103
           3       0.34      0.66      0.45       489
           4       0.17      0.03      0.05        66

    accuracy                           0.34      1405
   macro avg       0.27      0.22      0.20      1405
weighted avg       0.32      0.34      0.29      1405



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/mimic/alcohol_use_social_only.csv")
df = df[["note", "label"]].dropna()
df["label"] = df["label"].astype(int)  # ensure it's integer for classification

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["note"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)


In [ ]:
import torch

class AlcoholUseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} | {"labels": torch.tensor(self.labels[idx])}

    def __len__(self):
        return len(self.labels)

train_dataset = AlcoholUseDataset(train_encodings, train_labels)
val_dataset = AlcoholUseDataset(val_encodings, val_labels)

In [ ]:
!pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/mimic/alcohol_use_social_only.csv")
df = df[["note", "label"]].dropna()
df["label"] = df["label"].astype(int)

from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["note"].tolist(), df["label"].tolist(), test_size=0.2, stratify=df["label"], random_state=42
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256, return_tensors="pt")

In [ ]:
import torch
from torch.utils.data import Dataset

class AlcoholDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            key: val[idx] for key, val in self.encodings.items()
        } | {
            "labels": self.labels[idx]
        }

train_dataset = AlcoholDataset(train_encodings, train_labels)
val_dataset = AlcoholDataset(val_encodings, val_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification

num_classes = len(set(train_labels))
model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT", num_labels=num_classes
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from torch.utils.data import DataLoader
from torch import nn, optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)

optimizer = optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    model.train()
    total_loss = 0
    for i, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Epoch {epoch+1} | Batch {i} | Loss: {loss.item():.4f}")

    print(f"\n✅ Finished Epoch {epoch+1} | Total Loss: {total_loss:.4f}")

Epoch 1 | Batch 0 | Loss: 1.6348
Epoch 1 | Batch 20 | Loss: 1.4117
Epoch 1 | Batch 40 | Loss: 1.3955
Epoch 1 | Batch 60 | Loss: 2.1670


KeyboardInterrupt: 

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       331
           1       0.33      0.39      0.36       416
           2       0.00      0.00      0.00       103
           3       0.36      0.67      0.47       489
           4       0.00      0.00      0.00        66

    accuracy                           0.35      1405
   macro avg       0.14      0.21      0.17      1405
weighted avg       0.22      0.35      0.27      1405



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install checklist

  Using cached checklist-0.0.11.tar.gz (12.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached munch-4.0.0-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached patternfork_nosql-3.6.tar.gz (22.3 MB)
  Preparing metadata (setup.py) ... done
  Using cached iso-639-0.4.5.tar.gz (167 kB)
  Preparing metadata (setup.py) ... done
  Using cached backports.csv-1.0.7-py2.py3-none-any.whl.metadata (4.0 kB)
  Using cached feedparser-6.0.11-py3-none-any.whl.metadata (2.4 kB)
  Using cached pdfminer_six-20250416-py3-none-any.whl.metadata (4.1 kB)
  Using cached python_docx-1.1.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached CherryPy-18.10.0-py3-none-any.whl.metadata (8.7 kB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 349.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.8/349.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81

In [ ]:
from checklist.editor import Editor
from checklist.test_types import MFT
from checklist.test_suite import TestSuite
from checklist.perturb import Perturb
from checklist.pred_wrapper import PredictorWrapper

# Initialize editor
editor = Editor()

# Define substitution variables
verbs = ["drink", "consume"]
substances = ["alcohol", "ETOH", "wine", "beer"]
relatives = ["father", "sister", "cousin"]
times = ["3 years", "5 months", "a decade"]

# STEP 1: NEGATION (label = 0: Never)
negation_tests = editor.template(
    "Patient does not {verb} {substance}.",
    verb=verbs, substance=substances
)
negation_labels = [0] * len(negation_tests.data)

# STEP 2: ATTRIBUTION (label = 3: Current Every-Day)
attribution_tests = editor.template(
    "Patient drinks {substance}. {relation} does not drink {substance}.",
    relation=relatives, substance=substances
)
attribution_labels = [3] * len(attribution_tests.data)

# STEP 3: HISTORICAL (label = 1: Former)
history_tests = editor.template(
    "Patient used to drink {substance} but quit {time} ago.",
    substance=substances, time=times
)
history_labels = [1] * len(history_tests.data)


# STEP 5: Wrap all into a test suite
suite = TestSuite()
suite.add(MFT(data=negation_tests.data, labels=negation_labels, name="Negation", capability="Negation"))
suite.add(MFT(data=attribution_tests.data, labels=attribution_labels, name="Attribution", capability="Attribution"))
suite.add(MFT(data=history_tests.data, labels=history_labels, name="Historical", capability="Historical Phrases"))

In [ ]:
def predict_rf(texts):
    X = vectorizer.transform(texts)  # use your existing vectorizer
    return rf.predict(X)             # use your existing RandomForestClassifier

def predict_xgb(texts):
    X = vectorizer.transform(texts)
    return xgb.predict(X)            # use your existing XGBClassifier

# Wrap the predictors for CheckList
rf_wrapper = PredictorWrapper.wrap_predict(predict_rf)
xgb_wrapper = PredictorWrapper.wrap_predict(predict_xgb)

In [ ]:
#current
from checklist.pred_wrapper import PredictorWrapper

def predict_proba_rf(texts):
    X = vectorizer.transform(texts)  # Convert raw text to TF-IDF vectors
    return rf.predict_proba(X)       # Then get predicted probabilities

def predict_proba_xgb(texts):
    X = vectorizer.transform(texts)
    return xgb.predict_proba(X)

rf_wrapper = PredictorWrapper.wrap_softmax(predict_proba_rf)
xgb_wrapper = PredictorWrapper.wrap_softmax(predict_proba_xgb)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

def predict_bert(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
    return preds


bert_wrapper = PredictorWrapper.wrap_predict(predict_bert)

In [ ]:
def predict_rf_labels(texts):
    if not texts: # Handle empty input list if CheckList sends one
        return np.array([], dtype=int)
    X = vectorizer.transform(texts) # assuming vectorizer is defined
    predictions = rf.predict(X)     # assuming rf (RandomForestClassifier) is defined
    # wrap_predict expects an iterable of labels (list or numpy array)
    return np.array(predictions)

# Use wrap_predict for the Random Forest model
rf_wrapper = PredictorWrapper.wrap_predict(predict_rf_labels)

In [ ]:
print("📊 Running CheckList for Random Forest...")
suite.run(rf_wrapper, overwrite=True)
suite.summary()


📊 Running CheckList for Random Forest...
Running Negation
Predicting 8 examples
Running Attribution
Predicting 12 examples
Running Historical
Predicting 12 examples
Negation

Negation
Test cases:      8
Fails (rate):    8 (100.0%)

Example fails:
1 (0.2) Patient does not drink ETOH.
----
3 (0.2) Patient does not consume beer.
----
1 (0.2) Patient does not consume ETOH.
----




Historical Phrases

Historical
Test cases:      12
Fails (rate):    0 (0.0%)




Attribution

Attribution
Test cases:      12
Fails (rate):    6 (50.0%)

Example fails:
1 (0.2) Patient drinks ETOH. sister does not drink ETOH.
----
1 (0.2) Patient drinks ETOH. cousin does not drink ETOH.
----
1 (0.2) Patient drinks alcohol. father does not drink alcohol.
----






In [ ]:
# Run suite on XGBoost
print("📊 Running CheckList for XGBoost...")
suite.run(xgb_wrapper, overwrite=True)
suite.summary()


📊 Running CheckList for XGBoost...
Running Negation
Predicting 8 examples
Running Attribution
Predicting 12 examples
Running Historical
Predicting 12 examples
Negation

Negation
Test cases:      8
Fails (rate):    8 (100.0%)

Example fails:
3 (0.4) Patient does not consume wine.
----
1 (0.6) Patient does not consume alcohol.
----
1 (0.5) Patient does not consume ETOH.
----




Historical Phrases

Historical
Test cases:      12
Fails (rate):    0 (0.0%)




Attribution

Attribution
Test cases:      12
Fails (rate):    12 (100.0%)

Example fails:
1 (0.6) Patient drinks ETOH. cousin does not drink ETOH.
----
1 (0.6) Patient drinks ETOH. father does not drink ETOH.
----
1 (0.6) Patient drinks ETOH. sister does not drink ETOH.
----






In [ ]:
# Run suite on Bio_ClinicalBERT
print("📊 Running CheckList for Bio_ClinicalBERT...")
suite.run(bert_wrapper, overwrite=True)
suite.summary()


📊 Running CheckList for Bio_ClinicalBERT...
Running Negation
Predicting 8 examples
Running Attribution
Predicting 12 examples
Running Historical
Predicting 12 examples
Negation

Negation
Test cases:      8
Fails (rate):    8 (100.0%)

Example fails:
1 Patient does not drink alcohol.
----
1 Patient does not drink ETOH.
----
1 Patient does not drink beer.
----




Historical Phrases

Historical
Test cases:      12
Fails (rate):    0 (0.0%)




Attribution

Attribution
Test cases:      12
Fails (rate):    2 (16.7%)

Example fails:
1 Patient drinks ETOH. cousin does not drink ETOH.
----
1 Patient drinks alcohol. cousin does not drink alcohol.
----




